# Evaluation Analysis — Base vs SFT vs GRPO

Analyses `data/evaluation_results.csv` (500 test samples).

Columns: `expected`, `base_predicted`, `base_reward`, `sft_predicted`, `sft_reward`, `grpo_predicted`, `grpo_reward`

**Summary (500 test examples):**
| Model | Exact accuracy | Avg reward |
|-------|---------------|------------|
| Base Qwen3-8B | 25.2% | -0.340 |
| SFT | 89.2% | +0.787 |
| GRPO | 83.2% | +0.675 |

> GRPO regressed vs SFT in this run — the training config needs further tuning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('../data/evaluation_results.csv')

# Handle CSV with or without base model columns
HAS_BASE = 'base_reward' in df.columns
if not HAS_BASE:
    print("No base model columns found — using known baseline (0% accuracy, -1.0 reward)")

df['expected_action'] = df['expected'].str.split().str[0]
df['sft_action']      = df['sft_predicted'].str.split().str[0]
df['grpo_action']     = df['grpo_predicted'].str.split().str[0]
if HAS_BASE:
    df['base_action'] = df['base_predicted'].apply(
        lambda x: str(x).split()[0] if pd.notna(x) else None
    )

print(f'Rows: {len(df):,}  |  Has base model: {HAS_BASE}')
df.head()

## 1. Overall accuracy

In [ ]:
if HAS_BASE:
    base_acc = (df['base_reward'] == 1.0).mean() * 100
    base_avg = df['base_reward'].mean()
else:
    base_acc, base_avg = 0.0, -1.0

sft_acc  = (df['sft_reward']  == 1.0).mean() * 100
grpo_acc = (df['grpo_reward'] == 1.0).mean() * 100
sft_avg  = df['sft_reward'].mean()
grpo_avg = df['grpo_reward'].mean()

print(f'Base — exact accuracy: {base_acc:.1f}%  |  avg reward: {base_avg:.3f}')
print(f'SFT  — exact accuracy: {sft_acc:.1f}%  |  avg reward: {sft_avg:.3f}')
print(f'GRPO — exact accuracy: {grpo_acc:.1f}%  |  avg reward: {grpo_avg:.3f}')
print(f'\nSFT  vs Base: {sft_acc - base_acc:+.1f}%')
print(f'GRPO vs Base: {grpo_acc - base_acc:+.1f}%')
print(f'GRPO vs SFT : {grpo_acc - sft_acc:+.1f}%')

models = ['Base', 'SFT', 'GRPO']
accs   = [base_acc, sft_acc, grpo_acc]
colors = ['#aec6cf', 'steelblue', 'coral']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(models, accs, color=colors, width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 110)
ax.set_ylabel('Exact accuracy (%)')
ax.set_title('Base vs SFT vs GRPO — overall accuracy')
plt.tight_layout()
plt.show()

## 2. Reward distribution

In [ ]:
reward_vals = sorted(df['sft_reward'].unique())
sft_counts  = df['sft_reward'].value_counts().reindex(reward_vals, fill_value=0)
grpo_counts = df['grpo_reward'].value_counts().reindex(reward_vals, fill_value=0)

x = np.arange(len(reward_vals))
w = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
if HAS_BASE:
    base_counts = df['base_reward'].value_counts().reindex(reward_vals, fill_value=0)
    ax.bar(x - w, base_counts,  w, label='Base', color='#aec6cf')
    ax.bar(x,     sft_counts,   w, label='SFT',  color='steelblue')
    ax.bar(x + w, grpo_counts,  w, label='GRPO', color='coral')
else:
    ax.bar(x - w/2, sft_counts,  w, label='SFT',  color='steelblue')
    ax.bar(x + w/2, grpo_counts, w, label='GRPO', color='coral')

ax.set_xticks(x)
ax.set_xticklabels([str(v) for v in reward_vals])
ax.set_xlabel('Reward')
ax.set_ylabel('Count')
ax.set_title('Reward distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Accuracy by action type

In [ ]:
actions = ['check', 'call', 'fold', 'bet', 'raise']
base_by_action, sft_by_action, grpo_by_action, counts = [], [], [], []

for action in actions:
    mask = df['expected_action'] == action
    counts.append(mask.sum())
    sft_by_action.append((df.loc[mask, 'sft_reward']  == 1.0).mean() * 100)
    grpo_by_action.append((df.loc[mask, 'grpo_reward'] == 1.0).mean() * 100)
    if HAS_BASE:
        base_by_action.append((df.loc[mask, 'base_reward'] == 1.0).mean() * 100)
    else:
        base_by_action.append(0.0)

x = np.arange(len(actions))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w, base_by_action,  w, label='Base', color='#aec6cf')
ax.bar(x,     sft_by_action,   w, label='SFT',  color='steelblue')
ax.bar(x + w, grpo_by_action,  w, label='GRPO', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'{a}\n(n={c})' for a, c in zip(actions, counts)])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title('Accuracy by action type')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Where did SFT and GRPO improve / regress?

In [ ]:
df['grpo_vs_sft'] = df['grpo_reward'] - df['sft_reward']

improved  = (df['grpo_vs_sft'] > 0).sum()
regressed = (df['grpo_vs_sft'] < 0).sum()
same      = (df['grpo_vs_sft'] == 0).sum()

print(f'GRPO vs SFT:')
print(f'  Improved : {improved}  ({improved/len(df)*100:.1f}%)')
print(f'  Regressed: {regressed}  ({regressed/len(df)*100:.1f}%)')
print(f'  No change: {same}  ({same/len(df)*100:.1f}%)')

print('\n--- Cases where GRPO improved over SFT ---')
display(df[df['grpo_vs_sft'] > 0][['expected','sft_predicted','grpo_predicted','sft_reward','grpo_reward']].head(10))

print('\n--- Cases where GRPO regressed vs SFT ---')
display(df[df['grpo_vs_sft'] < 0][['expected','sft_predicted','grpo_predicted','sft_reward','grpo_reward']].head(10))

## 5. Confusion matrices

In [ ]:
from sklearn.metrics import confusion_matrix

cols = [('SFT', 'sft_action'), ('GRPO', 'grpo_action')]
if HAS_BASE:
    cols = [('Base', 'base_action')] + cols

fig, axes = plt.subplots(1, len(cols), figsize=(7 * len(cols), 5))
if len(cols) == 1:
    axes = [axes]

for ax, (label, col) in zip(axes, cols):
    valid = df[df[col].notna()]
    cm = confusion_matrix(valid['expected_action'], valid[col], labels=actions)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=actions, yticklabels=actions,
                cmap='Blues', ax=ax)
    ax.set_title(f'{label} — confusion matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Expected')

plt.tight_layout()
plt.show()

## 6. Bet/raise sizing accuracy

In [ ]:
sizing = df[df['expected_action'].isin(['bet', 'raise'])].copy()

def extract_amount(s):
    try:
        return float(str(s).split()[-1])
    except:
        return None

sizing['expected_amt'] = sizing['expected'].apply(extract_amount)
sizing['sft_amt']      = sizing['sft_predicted'].apply(extract_amount)
sizing['grpo_amt']     = sizing['grpo_predicted'].apply(extract_amount)

valid_sft  = sizing.dropna(subset=['expected_amt', 'sft_amt'])
valid_grpo = sizing.dropna(subset=['expected_amt', 'grpo_amt'])
sft_ratio  = valid_sft['sft_amt']  / valid_sft['expected_amt']
grpo_ratio = valid_grpo['grpo_amt'] / valid_grpo['expected_amt']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sft_ratio,  bins=40, alpha=0.6, label='SFT',  color='steelblue', range=(0, 3))
ax.hist(grpo_ratio, bins=40, alpha=0.6, label='GRPO', color='coral',     range=(0, 3))
ax.axvline(1.0, color='black', linestyle='--', label='Perfect sizing')
ax.set_xlabel('Predicted / Expected amount ratio')
ax.set_ylabel('Count')
ax.set_title('Bet/raise sizing ratio (1.0 = perfect)')
ax.legend()
plt.tight_layout()
plt.show()

within_10_sft  = ((sft_ratio  >= 0.9) & (sft_ratio  <= 1.1)).mean() * 100
within_10_grpo = ((grpo_ratio >= 0.9) & (grpo_ratio <= 1.1)).mean() * 100
print(f'Sizing within 10% — SFT: {within_10_sft:.1f}%  |  GRPO: {within_10_grpo:.1f}%')